# 07 — Hierarchical Risk Parity (HRP)

Notebooks 05 and 06 asked the same question of two different risk-based
objectives: the Most Diversified Portfolio (MDP, notebook 05) *kept* the
UUP corner that survives covariance shrinkage, and Equal Risk Contribution
(ERC, notebook 06) *shrank* it in relative terms without removing it. Both use the full Σ — cross-covariances included — in the condition that defines their weights (MDP's first-order condition, ERC's equal-risk-contribution equations).

This notebook asks the same question of a hierarchy-based method that never
inverts &Sigma; at all. Hierarchical Risk Parity (HRP) clusters assets by
correlation distance and allocates capital by walking down the resulting
tree, splitting mass at each node in inverse proportion to the variance of
the two sides' inverse-variance sub-portfolios. What happens to the UUP
corner under a method that only ever compares diagonal blocks of &Sigma;,
and does UUP sit in a cluster of its own?

## §1 Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize as opt
import scipy.cluster.hierarchy as sch
from scipy.cluster.hierarchy import dendrogram, is_valid_linkage, is_monotonic, cophenet
from scipy.spatial.distance import pdist, squareform

import maplab as ml
from maplab import (
    Panel, apply_style, GMV, MaxSharpe, EqualWeight, MostDiversified,
    EqualRiskContribution, HierarchicalRiskParity, risk_contributions,
    backtest, summary, mdp_closed_form,
)
from maplab.models import (
    _min_variance_long_only, _mdp_long_only, _erc_long_only, _tangency_long_only,
    _hrp_long_only,
)

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

print(f"prices: {prices.shape[0]:,} days x {prices.shape[1]} assets "
      f"({prices.index.min().date()} -> {prices.index.max().date()})")
print(f"train/test split: {split_ts.date()}")

rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"
print(f"rebalance labels: n={len(rdates)}  first={rdates[0].date()}  last={rdates[-1].date()}")

In [ ]:
# Copied from notebook 04/05/06 (notebook-local helper, not part of the shared harness).
def ols_with_intercept(y: pd.Series, X: pd.DataFrame) -> dict:
    y_np = y.to_numpy()
    X_np = X.to_numpy()
    n = len(y_np)
    Xd = np.column_stack([np.ones(n), X_np])
    coefs, _, _, _ = np.linalg.lstsq(Xd, y_np, rcond=None)
    fitted = Xd @ coefs
    resid = y_np - fitted
    dof = n - Xd.shape[1]
    sigma2 = float(resid @ resid) / dof
    XtX_inv = np.linalg.inv(Xd.T @ Xd)
    se = np.sqrt(np.diag(XtX_inv) * sigma2)
    tstats = coefs / se
    ss_tot = float(np.sum((y_np - y_np.mean()) ** 2))
    ss_res = float(resid @ resid)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    names = ["alpha"] + list(X.columns)
    return {
        "coef": pd.Series(coefs, index=names),
        "tstat": pd.Series(tstats, index=names),
        "r2": r2,
        "n": n,
    }


def half_l1(w1, w2) -> float:
    return float(0.5 * np.abs(np.asarray(w1) - np.asarray(w2)).sum())


def effN(w) -> float:
    w = np.asarray(w)
    return float(1.0 / (w ** 2).sum())


def port_vol(w, Sigma_np) -> float:
    w = np.asarray(w)
    return float(np.sqrt(w @ Sigma_np @ w))


def dr(w, sigma_np, Sigma_np) -> float:
    w = np.asarray(w)
    return float(w @ sigma_np) / port_vol(w, Sigma_np)


def n_uncorr(w, Sigma_np) -> float:
    # Exp of the entropy of p_k = (e_k'w)^2 lambda_k / w'Sigma w, eigh(Sigma), lambda clipped at 0.
    w = np.asarray(w)
    lam, E = np.linalg.eigh(Sigma_np)
    lam = np.clip(lam, 0.0, None)
    v = (E.T @ w) ** 2 * lam
    total = v.sum()
    p = v / total
    p = p[p > 0]
    return float(np.exp(-np.sum(p * np.log(p))))


def max_rc_share(w, Sigma_np) -> float:
    w = np.asarray(w)
    rc = w * (Sigma_np @ w)
    return float(np.max(rc) / rc.sum())


def ivp_var_independent(Sigma_np, idx) -> float:
    # Variance of the inverse-variance sub-portfolio on idx, written out here
    # (not via _hrp_cluster_var) so mechanism checks are an independent check.
    idx = list(idx)
    sub = Sigma_np[np.ix_(idx, idx)]
    p = 1.0 / np.diag(sub)
    p = p / p.sum()
    return float(p @ sub @ p)


def sigma_cf_zero_corr(Sigma_np, i):
    # Sigma with asset i's off-diagonal correlations set to 0 (its own variance kept).
    sigma = np.sqrt(np.diag(Sigma_np))
    C = Sigma_np / np.outer(sigma, sigma)
    C = 0.5 * (C + C.T)
    C[i, :] = 0.0
    C[:, i] = 0.0
    C[i, i] = 1.0
    return np.outer(sigma, sigma) * C


def root_singleton(Z, i) -> bool:
    return i in (int(Z[-1, 0]), int(Z[-1, 1]))

In [ ]:
class InverseVariance(ml.Strategy):
    # Notebook-local: inverse-variance weights, sample covariance window.
    # w propto 1/diag(Sigma) -- HRP's reference case (HRP on diag(Sigma) = IVP).
    name = "IVP"
    family = "Risk-based"

    def predict_weights(self, panel: Panel, asof: pd.Timestamp) -> pd.Series:
        _, Sigma = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof)
        diag = np.diag(Sigma.to_numpy())
        w = (1.0 / diag) / np.sum(1.0 / diag)
        return pd.Series(w, index=Sigma.columns)

### §1.2 Recipe and identities

**Recipe.** C = D&#8315;&#185;&Sigma;D&#8315;&#185; (the correlation matrix);
d<sub>ij</sub> = &#8730;((1&#8722;&rho;<sub>ij</sub>)/2); cluster on the
Euclidean distance between COLUMNS of d (distance of distances, the
literature's recipe); single linkage; then recursive bisection. At every
split the two sides get capital in inverse proportion to the variance of
each side's inverse-variance sub-portfolio:
&alpha;<sub>L</sub> = V<sub>R</sub>/(V<sub>L</sub>+V<sub>R</sub>), V = w'&Sigma;w
for w = the IVP weights of that side alone. **Registered bisection is
tree-following** (split at each dendrogram node, top to bottom). The
literature's original algorithm instead splits the LEAF ORDER into equal
halves at each step ("positional"); a read-only check run before this
notebook found that positional HRP's weights change with the input column
order on 182/208 rebalances (&gt; 1% half-L1 on 116 of them), which is why
tree-following is registered here and positional is carried only as a
sensitivity.

Only the diagonal blocks &Sigma;<sub>LL</sub>, &Sigma;<sub>RR</sub> ever
enter an allocation decision; the cross block &Sigma;<sub>LR</sub> never
does — it only shapes which assets end up together in the tree.

**Identities.**
- C = I (Sigma diagonal) &#8658; HRP = inverse variance = long-only GMV,
  for any tree and either bisection. Contrast: on the same diagonal
  &Sigma;, ERC and MDP both reduce to inverse VOL, not inverse variance —
  HRP is a variance-based method, ERC/MDP are vol-based.
- N = 2 &#8658; inverse variance for any &rho;.
- Exchangeable &Sigma; (equal vols, constant &rho;) &#8658; GMV = MDP = ERC
  = EW, but positional HRP &#8800; EW for &rho; &gt; 0 whenever N is not a
  power of 2 — the halves split is size-dependent even when every asset is
  otherwise identical.
- Uniform scaling c·Σ leaves HRP's weights unchanged. Per-asset scaling KΣK leaves the tree unchanged (correlations are unchanged) but not the weights in any simple way: HRP has no analogue of ERC/MDP's rule w(KΣK) ∝ K⁻¹w(Σ).
- **Root-singleton identity.** If an asset sits alone on one side of the FINAL merge, its weight is exactly V<sub>rest</sub>/(σ<sup>2</sup> &#43; V<sub>rest</sub>), independent of its correlations with the rest of the universe — HRP is "correlation-blind" for that asset in exactly this sense.

## §2 Pre-registration (written before any backtest output)

**Registered:** HRP(sample) = tree-following bisection, single linkage, distance of distances, inverse variance within clusters. **Reproduction gate:** GMV(sample), EW, MDP(sample), ERC(sample). **Sensitivities:** HRP(ledoit_wolf); HRP&#91;positional&#93;(sample), the literature's equal-halves bisection; HRP&#91;ward&#93;(sample), selected by the pre-set recipe rule. **Reference:** IVP (inverse variance, notebook-local) = HRP on diag(Σ).

**Windows:** full / train (&lt;= 2022-12-31) / test (2023+). Paired test = daily return difference, t = mean/sd·√n, plain OLS, no HAC; counts only at &#124;t&#124; &gt;= 2. Count-based hypotheses use target weights at the 208 rebalances (40 in test).

Directions for H1a, H2b and H3 were fixed by a read-only snapshot check at 2014-12-31 / 2017-12-31 / 2022-12-31, run before this notebook on sample-covariance solutions at those three dates only (pre-set rule: a direction is registered only if it holds at 3/3 dates). H7's direction is fixed a priori by theory, not by the snapshot.

**Count rule** (H1a, H2b, H3): SUPPORTED iff the inequality holds at 2022-12-31 AND on more than 50% of the 208 rebalances AND on more than 50% of the 40 test rebalances. If it holds at 2022-12-31 and on more than 50% of the 208 but not on more than 50% of the test rebalances: NOT ROBUST. Otherwise: NOT SUPPORTED.

**H1a (UUP corner):** w_UUP(HRP) &lt; w_UUP(GMV) AND w_UUP(HRP) &lt; w_UUP(MDP), jointly. **H1b:** HRP vs ERC was mixed at the snapshot, so no direction is registered; the count is descriptive.

**H2a (descriptive):** number of rebalances at which UUP is a root singleton (alone on one side of the final merge). **H2b (correlation blindness):** with Σ_cf = Σ with UUP's correlations set to 0, &#124;w_UUP(Σ) − w_UUP(Σ_cf)&#124; for HRP &lt; the same quantity for GMV, for MDP and for ERC, each comparator judged separately by the count rule. Whenever UUP is a root singleton under both Σ and Σ_cf, HRP's difference is exactly 0 by construction; that count is reported next to H2b and is its empirical content.

**H3 (effN):** effN(HRP) &gt; effN(GMV); effN(HRP) &gt; effN(MDP); effN(HRP) &lt; effN(ERC), each judged separately by the count rule.

**H4 (nulls):** HRP − GMV, HRP − MDP, HRP − ERC: Δα from the direct diff-regression, &#124;t&#124; &lt; 2. Total paired differences are reported with the β decomposition and carry no directional claim — the methods differ in ex-ante vol on the same Σ.

**H5 (HRP − EW &lt; 0, beta-driven):** SUPPORTED iff the full-window paired t &lt;= −2 AND the test-window mean &lt; 0 AND &#124;Δβ part&#124; &gt; &#124;Δα&#124;.

**H6 (sensitivity nulls):** HRP(LW) − HRP(S): total paired difference, &#124;t&#124; &lt; 2 on full AND test. HRP − IVP, HRP[positional] − HRP, HRP[ward] − HRP: Δα, &#124;t&#124; &lt; 2.

**H7 (forecast bias vs GMV):** q = log(realized holding-segment vol / ex-ante √(w′Σw)) at each rebalance. mean(q_HRP − q_GMV) &lt; 0: SUPPORTED iff t &lt;= −2 over the full-window segments AND the test-window mean &lt; 0; full-window t &lt;= −2 alone = NOT ROBUST. Rationale: GMV's weights minimize in-sample variance, so its ex-ante vol is biased low; HRP optimizes nothing. MDP, ERC, IVP and EW biases are descriptive.

**Wording:** a null with 1.5 &lt;= &#124;t&#124; &lt; 2 holds "narrowly".

**Honest limitations:** the snapshot dates are in-sample (2022-12-31 is inside the train window and is backtested here); H2b is partly mechanical (see above); at the snapshot the recipe was not robust to the linkage choice (average / ward / direct moved weights by 7–17% half-L1), and only ward is backtested.

## §3 Snapshot at three dates (sample Σ)

Recomputed at 2014-12-31 / 2017-12-31 / 2022-12-31 — the same three dates
that fixed the H1a/H2b/H3 directions in the read-only design check. §3.1:
what do the weights actually look like?

In [ ]:
DATES = [pd.Timestamp(d) for d in ["2014-12-31", "2017-12-31", "2022-12-31"]]

def comparators_at(Sigma, asof, tag):
    w_gmv = _min_variance_long_only(Sigma)
    w_mdp, _ = _mdp_long_only(Sigma, tag, asof)
    w_erc, _ = _erc_long_only(Sigma, tag, asof)
    return {"GMV": np.asarray(w_gmv, float), "MDP": np.asarray(w_mdp, float), "ERC": np.asarray(w_erc, float)}

snap = {}
for asof in DATES:
    _, Sigma = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof)
    cols = list(Sigma.columns)
    Sigma_np = Sigma.to_numpy()
    n = len(cols)
    iU = cols.index("UUP")
    sigma_np = np.sqrt(np.diag(Sigma_np))
    C = Sigma_np / np.outer(sigma_np, sigma_np)

    w_hrp, info = _hrp_long_only(Sigma, "snap", asof, bisection="tree")
    w_pos, info_pos = _hrp_long_only(Sigma, "snap", asof, bisection="positional")
    w_ivp = (1.0 / np.diag(Sigma_np)) / np.sum(1.0 / np.diag(Sigma_np))
    comps = comparators_at(Sigma, asof, "snap")
    w_ew = np.full(n, 1.0 / n)

    W = {"HRP": w_hrp, "HRP[pos]": w_pos, "IVP": w_ivp, **comps, "EW": w_ew}

    Sigma_cf_np = sigma_cf_zero_corr(Sigma_np, iU)
    Sigma_cf = pd.DataFrame(Sigma_cf_np, index=cols, columns=cols)
    w_hrp_cf, info_cf = _hrp_long_only(Sigma_cf, "snapcf", asof, bisection="tree")
    comps_cf = comparators_at(Sigma_cf, asof, "snapcf")

    Z = info["Z"]
    # Rebuild D/y explicitly (the registered recipe: half-formula, distance of distances)
    # to compute the cophenetic correlation and to quasi-diagonalize the heatmap in F2.
    C_half = np.clip(0.5 * (C + C.T), -1.0, 1.0)
    np.fill_diagonal(C_half, 1.0)
    D = np.sqrt(0.5 * (1.0 - C_half))
    np.fill_diagonal(D, 0.0)
    y = pdist(D, metric="euclidean")

    alts = {
        "average": _hrp_long_only(Sigma, "snap", asof, linkage="average")[0],
        "ward": _hrp_long_only(Sigma, "snap", asof, linkage="ward")[0],
        "direct": _hrp_long_only(Sigma, "snap", asof, dist_of_dist=False)[0],
    }

    # sqrt(1-rho) sanity: rebuild d = sqrt(1-rho) directly and re-cluster/bisect in-notebook.
    C_clip = np.clip(0.5 * (C + C.T), -1.0, 1.0)
    np.fill_diagonal(C_clip, 1.0)
    D_full = np.sqrt(1.0 - C_clip)
    np.fill_diagonal(D_full, 0.0)
    y_full = pdist(D_full, metric="euclidean")
    Z_full = sch.linkage(y_full, method="single")
    w_sanity = np.ones(n)
    stack = [sch.to_tree(Z_full)]
    while stack:
        node = stack.pop()
        if node.is_leaf():
            continue
        left, right = node.get_left(), node.get_right()
        L, R = left.pre_order(), right.pre_order()
        vL, vR = ivp_var_independent(Sigma_np, L), ivp_var_independent(Sigma_np, R)
        a = vR / (vL + vR)
        w_sanity[L] *= a
        w_sanity[R] *= 1.0 - a
        stack += [left, right]

    snap[asof] = dict(
        cols=cols, Sigma_np=Sigma_np, sigma_np=sigma_np, C=C, iU=iU, n=n,
        W=W, comps=comps, w_ivp=w_ivp, w_ew=w_ew,
        info=info, info_pos=info_pos, info_cf=info_cf, D=D, y=y,
        Sigma_cf_np=Sigma_cf_np, comps_cf=comps_cf, w_hrp_cf=w_hrp_cf,
        alts=alts, w_sanity=w_sanity,
    )

print("snapshot computed at:", [d.date() for d in DATES])

In [ ]:
F4 = lambda v: f"{v:.4f}"
E3 = lambda v: f"{v:.3e}"

for asof in DATES:
    s = snap[asof]
    print(f"\n----- {asof.date()} weights -----")
    tbl = pd.DataFrame(s["W"], index=s["cols"])
    print(tbl.to_string(float_format=F4))

§3.2 — diversification: does HRP raise effN and lower the UUP weight
relative to GMV / MDP / ERC, and where does UUP sit in the correlation /
vol structure at each date?

In [ ]:
for asof in DATES:
    s = snap[asof]
    Sigma_np, iU, n = s["Sigma_np"], s["iU"], s["n"]
    C = s["C"]
    rho_bar = float((C.sum() - n) / (n * (n - 1)))
    uup_mean_corr = float((C[iU].sum() - 1.0) / (n - 1))
    uup_vol_rank = int((s["sigma_np"] <= s["sigma_np"][iU]).sum())
    print(f"\n----- {asof.date()} -----")
    print(f"UUP mean pairwise corr={F4(uup_mean_corr)}  vol rank (1=lowest)={uup_vol_rank}  rho_bar={F4(rho_bar)}")

    rows = {}
    for name, w in s["W"].items():
        rows[name] = {
            "w_UUP": float(w[iU]), "effN": effN(w), "n_uncorr": n_uncorr(w, Sigma_np),
            "max_RC_share": max_rc_share(w, Sigma_np), "DR": dr(w, s["sigma_np"], Sigma_np),
            "exante_vol": port_vol(w, Sigma_np),
        }
    div_tbl = pd.DataFrame(rows).T
    print(div_tbl.to_string(float_format=F4))

§3.3 — tree facts: is UUP alone on one side of the final merge, and how
much of the total split cascades onto a single asset?

In [ ]:
def tree_path(Sigma_np, Z, i):
    node = sch.to_tree(Z)
    path = []
    while not node.is_leaf():
        L = node.get_left().pre_order(); R = node.get_right().pre_order()
        vL, vR = ivp_var_independent(Sigma_np, L), ivp_var_independent(Sigma_np, R)
        aL = vR / (vL + vR)
        if i in L:
            path.append((len(L), len(R), aL))
            node = node.get_left()
        else:
            path.append((len(R), len(L), 1.0 - aL))
            node = node.get_right()
    return path


def pos_path(Sigma_np, order, i):
    it = list(order)
    path = []
    while len(it) > 1:
        h = len(it) // 2
        L, R = it[:h], it[h:]
        vL, vR = ivp_var_independent(Sigma_np, L), ivp_var_independent(Sigma_np, R)
        aL = vR / (vL + vR)
        if i in L:
            path.append((len(L), len(R), aL))
            it = L
        else:
            path.append((len(R), len(L), 1.0 - aL))
            it = R
    return path


for asof in DATES:
    s = snap[asof]
    Sigma_np, iU, cols = s["Sigma_np"], s["iU"], s["cols"]
    Z = s["info"]["Z"]
    coph = float(cophenet(Z, s["y"])[0])
    print(f"\n----- {asof.date()} -----")
    print(f"UUP root singleton (tree): {root_singleton(Z, iU)}   cophenetic corr={F4(coph)}   "
          f"last merge height={F4(Z[-1, 2])}")
    tp = tree_path(Sigma_np, Z, iU)
    tp_display = [(a, b, round(c, 4)) for a, b, c in tp]
    print("UUP tree path (own-side n, other-side n, share to own side):", tp_display)
    prod = 1.0
    for _, _, a in tp:
        prod *= a
    print(f"  |path product - w_UUP| = {E3(abs(prod - s['W']['HRP'][iU]))}")
    order = s["info"]["order"]
    pp = pos_path(Sigma_np, s["info_pos"]["order"], iU)
    pp_display = [(a, b, round(c, 4)) for a, b, c in pp]
    print("UUP positional path (own-side n, other-side n, share to own side):", pp_display,
          f" isolation depth={len(pp)}")
    print("leaf order (tree):", [cols[j] for j in order])

§3.4 — counterfactual: with UUP's correlations set to 0, does its weight
move, and by how much relative to GMV / MDP / ERC?

In [ ]:
for asof in DATES:
    s = snap[asof]
    iU = s["iU"]
    Z_cf = s["info_cf"]["Z"]
    dcf = {"HRP": abs(s["W"]["HRP"][iU] - s["w_hrp_cf"][iU])}
    dcf.update({m: abs(s["comps"][m][iU] - s["comps_cf"][m][iU]) for m in s["comps"]})
    print(f"\n----- {asof.date()} -----  UUP root singleton under Sigma_cf: {root_singleton(Z_cf, iU)}")
    cf_tbl = pd.DataFrame({
        "w_UUP": {"HRP": s["W"]["HRP"][iU], **{m: s["comps"][m][iU] for m in s["comps"]}},
        "w_UUP_cf": {"HRP": s["w_hrp_cf"][iU], **{m: s["comps_cf"][m][iU] for m in s["comps"]}},
        "abs_delta": dcf,
    })
    print(cf_tbl.to_string(float_format=F4))

§3.5 — recipe sensitivity: how much do the weights move under a different
linkage (average / ward), a direct distance (no distance-of-distances), or
d = √(1−ρ) instead of √((1−ρ)/2) (a sanity check that should show 0
movement, since it just rescales every pairwise distance by a constant)?

In [ ]:
for asof in DATES:
    s = snap[asof]
    w_hrp = s["W"]["HRP"]
    rec = {"alt_" + k: half_l1(v, w_hrp) for k, v in s["alts"].items()}
    rec["sqrt(1-rho)_sanity"] = half_l1(s["w_sanity"], w_hrp)
    rec["tree_vs_positional"] = half_l1(s["W"]["HRP[pos]"], w_hrp)
    print(f"\n----- {asof.date()} (half-L1 vs registered HRP) -----")
    print(pd.Series(rec).to_string(float_format=E3))

### Figure F1 — snapshot map at 2022-12-31

Where do HRP, GMV, MDP, ERC, MaxSharpe and EW sit in (vol, μ) space at the
train-window snapshot that fixed the H1a/H2b/H3 directions? μ here is a
252-day trailing sample mean, so the y-axis is illustrative — the honest
reading is the x-position (risk) and the printed weights, not "expected
return". MaxSharpe is computed at this single date only (snapshot reference, not backtested); IVP is backtested in §4.

In [ ]:
asof_2022 = pd.Timestamp("2022-12-31")
mu_2022, Sigma_2022 = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_2022)
Sigma_np = Sigma_2022.to_numpy()
mu_np = mu_2022.to_numpy()
n = Sigma_np.shape[0]
tickers = list(Sigma_2022.columns)

s2022 = snap[asof_2022]
w_hrp_2022 = s2022["W"]["HRP"]
w_gmv_2022 = s2022["W"]["GMV"]
w_mdp_2022 = s2022["W"]["MDP"]
w_erc_2022 = s2022["W"]["ERC"]
w_ivp_2022 = s2022["W"]["IVP"]
w_ew_2022 = s2022["W"]["EW"]

rf_ann_2022 = MaxSharpe()._rf_ann(panel, asof_2022)
excess_2022 = mu_2022 - rf_ann_2022
if excess_2022.max() > 0:
    w_msr_2022, _ = _tangency_long_only(excess_2022.to_numpy(), Sigma_np, "F1", asof_2022)
else:
    w_msr_2022 = _min_variance_long_only(Sigma_2022)

def min_variance_at_target(Sigma_np, mu_np, target, x0):
    n = Sigma_np.shape[0]
    ones = np.ones(n)
    cons = [
        {"type": "eq", "fun": lambda w: w.sum() - 1.0, "jac": lambda w: ones},
        {"type": "eq", "fun": lambda w: w @ mu_np - target, "jac": lambda w: mu_np},
    ]
    bounds = [(0.0, 1.0)] * n
    res = opt.minimize(
        lambda w: w @ Sigma_np @ w, x0, jac=lambda w: 2 * Sigma_np @ w, method="SLSQP",
        bounds=bounds, constraints=cons, options={"ftol": 1e-12, "maxiter": 1000},
    )
    return res.x if res.success else None

rng_mc = np.random.default_rng(7)
mc_weights = rng_mc.dirichlet(np.full(n, 0.3), size=20_000)
mc_ret = mc_weights @ mu_np
mc_vol = np.sqrt(np.einsum("ij,jk,ik->i", mc_weights, Sigma_np, mc_weights))

targets = np.linspace(mu_np.min(), mu_np.max(), 60)
x0 = np.full(n, 1.0 / n)
frontier_ret, frontier_vol = [], []
for t in targets:
    w = min_variance_at_target(Sigma_np, mu_np, t, x0)
    if w is None:
        continue
    x0 = w
    frontier_ret.append(w @ mu_np)
    frontier_vol.append(np.sqrt(w @ Sigma_np @ w))
frontier_ret = np.array(frontier_ret)
frontier_vol = np.array(frontier_vol)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(mc_vol, mc_ret, s=2, alpha=0.15, color="#bbbbbb", zorder=0,
           label="random long-only portfolios (Dirichlet a=0.3)")

asset_vol = np.sqrt(np.diag(Sigma_np))
group_order = list(ml.ASSET_GROUPS.keys())
group_colors = dict(zip(group_order, plt.cm.tab10(np.linspace(0, 1, len(group_order)))))
for i, tkr in enumerate(tickers):
    grp = ml.GROUP_OF[tkr]
    is_uup = tkr == "UUP"
    ax.scatter(asset_vol[i], mu_np[i], s=30, color=group_colors[grp], zorder=5 if is_uup else 2)
    ax.annotate(tkr, (asset_vol[i], mu_np[i]), fontsize=6,
                xytext=(3, 3), textcoords="offset points",
                color="#c0392b" if is_uup else "#666666",
                fontweight="bold" if is_uup else "normal",
                zorder=5 if is_uup else 2)

hrp_ret = float(w_hrp_2022 @ mu_np); hrp_vol = port_vol(w_hrp_2022, Sigma_np)
gmv_ret = float(w_gmv_2022 @ mu_np); gmv_vol = port_vol(w_gmv_2022, Sigma_np)
mdp_ret = float(w_mdp_2022 @ mu_np); mdp_vol = port_vol(w_mdp_2022, Sigma_np)
erc_ret = float(w_erc_2022 @ mu_np); erc_vol = port_vol(w_erc_2022, Sigma_np)
msr_ret = float(w_msr_2022 @ mu_np); msr_vol = port_vol(w_msr_2022, Sigma_np)
ew_ret = float(w_ew_2022 @ mu_np); ew_vol = port_vol(w_ew_2022, Sigma_np)
ivp_ret = float(w_ivp_2022 @ mu_np); ivp_vol = port_vol(w_ivp_2022, Sigma_np)

efficient = frontier_ret >= gmv_ret
ax.plot(frontier_vol[~efficient], frontier_ret[~efficient], color="#999999", ls="--", lw=1.2, zorder=3)
ax.plot(frontier_vol[efficient], frontier_ret[efficient], color="#999999", lw=1.8, zorder=3, label="long-only frontier")

ax.scatter([hrp_vol], [hrp_ret], marker="o", s=170, color="#8e44ad", zorder=4, label="HRP")
ax.scatter([gmv_vol], [gmv_ret], marker="*", s=200, color=ml.FAMILY_COLORS["Risk-based"], zorder=4, label="GMV")
ax.scatter([mdp_vol], [mdp_ret], marker="P", s=140, color=ml.FAMILY_COLORS["Risk-based"], zorder=4, label="MDP")
ax.scatter([erc_vol], [erc_ret], marker="X", s=160, color="#d4a017", zorder=4, label="ERC")
ax.scatter([msr_vol], [msr_ret], marker="*", s=200, color=ml.FAMILY_COLORS["Return-based"], zorder=4, label="MaxSharpe")
ax.scatter([ew_vol], [ew_ret], marker="D", s=90, color=ml.FAMILY_COLORS["Benchmark"], zorder=4, label="EqualWeight")
ax.scatter([ivp_vol], [ivp_ret], marker="^", s=110, color="#555555", zorder=4, label="IVP")

ax.set_xlabel("annualized volatility")
ax.set_ylabel("annualized arithmetic expected return (mu)")
ax.set_title(f"Snapshot {asof_2022.date()} -- HRP / GMV / MDP / ERC / MaxSharpe / EW / IVP")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8, borderaxespad=0)
plt.tight_layout(); plt.show()

print(f"w_UUP: HRP={w_hrp_2022[s2022['iU']]:.4f}  GMV={w_gmv_2022[s2022['iU']]:.4f}  "
      f"MDP={w_mdp_2022[s2022['iU']]:.4f}  ERC={w_erc_2022[s2022['iU']]:.4f}  "
      f"MaxSharpe={w_msr_2022[s2022['iU']]:.4f}  EW={w_ew_2022[s2022['iU']]:.4f}  "
      f"IVP={w_ivp_2022[s2022['iU']]:.4f}")

### Figure F2 — mechanism grid: dendrogram, quasi-diagonalized correlation, UUP weight bars

Three rows (the three snapshot dates) by three panels: (a) the dendrogram
with UUP's leaf label highlighted and the root split's share to UUP's side
annotated; (b) the correlation matrix reordered into the dendrogram's leaf
order (quasi-diagonalized), with UUP's row/column boxed and a line at the
root split boundary; (c) UUP's weight under IVP / HRP / HRP[pos] / ERC /
GMV / MDP / EW, with hollow bars showing the same methods' weight under
Σ_cf (UUP's correlations zeroed). A small separate panel checks the split
identity directly: mass ratio vs. V_R/(V_L+V_R) at every dendrogram node,
every snapshot date.

(a) now shows the capital cascade: the share each node passes to its left &#124; right branch, and each leaf's final HRP weight.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(17, 13))
uup_methods = ["IVP", "HRP", "HRP[pos]", "ERC", "GMV", "MDP", "EW"]
cf_methods = ["HRP", "GMV", "MDP", "ERC"]

for r, asof in enumerate(DATES):
    s = snap[asof]
    cols, iU, Z, order = s["cols"], s["iU"], s["info"]["Z"], s["info"]["order"]

    # (a) dendrogram -- capital cascade: leaf labels show final HRP weight,
    # internal nodes show the L | R share of HRP's tree split at that node.
    ax = axes[r, 0]
    labels = np.array(cols)
    dend = dendrogram(Z, labels=labels, ax=ax, leaf_rotation=90, leaf_font_size=6,
                       color_threshold=0, above_threshold_color="#555555")
    w_hrp_map = {c: s["W"]["HRP"][ci] for ci, c in enumerate(cols)}
    new_labels = [f"{t}\n{w_hrp_map[t]:.1%}" for t in dend["ivl"]]
    ax.set_xticklabels(new_labels, rotation=90, fontsize=6)
    for lbl in ax.get_xticklabels():
        if lbl.get_text().startswith("UUP"):
            lbl.set_color("#c0392b"); lbl.set_fontweight("bold")
    root_node, nodelist = sch.to_tree(Z, rd=True)
    root_L = root_node.get_left().pre_order()
    root_R = root_node.get_right().pre_order()
    v_l = ivp_var_independent(s["Sigma_np"], root_L)
    v_r = ivp_var_independent(s["Sigma_np"], root_R)
    a_uup_side = v_r / (v_l + v_r) if iU in root_L else v_l / (v_l + v_r)
    ax.set_title(f"{asof.date()} dendrogram (root share to UUP side={a_uup_side:.3f})", fontsize=8)

    leaves_order = dend["leaves"]
    leaf_pos = {leaf: pos for pos, leaf in enumerate(leaves_order)}
    icoord = np.array(dend["icoord"])
    dcoord = np.array(dend["dcoord"])
    n_links = Z.shape[0]
    annotated = 0
    for k in range(icoord.shape[0]):
        h = dcoord[k, 1]
        x_pos = float(np.mean(icoord[k, 1:3]))
        matches = np.where(np.abs(Z[:, 2] - h) <= 1e-12)[0]
        if len(matches) != 1:
            continue  # ambiguous tied-height link, skip
        j = int(matches[0])
        left_id, right_id = int(Z[j, 0]), int(Z[j, 1])
        left_leaves = nodelist[left_id].pre_order()
        right_leaves = nodelist[right_id].pre_order()
        left_positions = [leaf_pos[l] for l in left_leaves]
        right_positions = [leaf_pos[l] for l in right_leaves]
        assert max(left_positions) < min(right_positions), (
            f"{asof.date()}: link {k} (row {j}) -- plotted left child is not Z[j,0]"
        )
        left_set = set(left_leaves)
        split_shares = None
        for (L, R, a) in info["splits"]:
            if set(L) == left_set:
                split_shares = (a, 1.0 - a)
                break
            if set(R) == left_set:
                split_shares = (1.0 - a, a)
                break
        if split_shares is None:
            continue
        a_left, a_right = split_shares
        ax.annotate(f"{a_left:.2f} | {a_right:.2f}", (x_pos, h), fontsize=5,
                    ha="center", va="bottom", color="#2c3e50",
                    xytext=(0, 2 + 6 * (k % 2)), textcoords="offset points")
        annotated += 1
    print(f"{asof.date()}: annotated {annotated}/{n_links} nodes")

    # (b) quasi-diagonalized correlation heatmap
    ax = axes[r, 1]
    C_ord = s["C"][np.ix_(order, order)]
    im = ax.imshow(C_ord, vmin=-1, vmax=1, cmap="RdBu_r")
    ordered_labels = [cols[j] for j in order]
    pos_uup = ordered_labels.index("UUP")
    ax.axhline(pos_uup - 0.5, color="black", lw=0.6)
    ax.axhline(pos_uup + 0.5, color="black", lw=0.6)
    ax.axvline(pos_uup - 0.5, color="black", lw=0.6)
    ax.axvline(pos_uup + 0.5, color="black", lw=0.6)
    boundary = len(root_L) - 0.5
    ax.axhline(boundary, color="#c0392b", lw=1.2, ls="--")
    ax.axvline(boundary, color="#c0392b", lw=1.2, ls="--")
    ax.set_xticks(range(s["n"])); ax.set_xticklabels(ordered_labels, rotation=90, fontsize=5)
    ax.set_yticks(range(s["n"])); ax.set_yticklabels(ordered_labels, fontsize=5)
    ax.set_title(f"{asof.date()} correlation (leaf order)", fontsize=8)

    # (c) UUP weight bars, hollow = counterfactual
    ax = axes[r, 2]
    x = np.arange(len(uup_methods))
    vals = [s["W"][m][iU] for m in uup_methods]
    ax.bar(x, vals, color=["#555555" if m != "HRP" else "#8e44ad" for m in uup_methods])
    for m in cf_methods:
        xi = uup_methods.index(m)
        cf_val = s["w_hrp_cf"][iU] if m == "HRP" else s["comps_cf"][m][iU]
        ax.bar(xi, cf_val, facecolor="none", edgecolor="#c0392b", lw=1.5, hatch="//")
    ax.set_xticks(x); ax.set_xticklabels(uup_methods, rotation=45, fontsize=7, ha="right")
    ax.set_title(f"{asof.date()} UUP weight (hollow = Sigma_cf)", fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for asof in DATES:
    s = snap[asof]
    Sigma_np = s["Sigma_np"]
    mass_ratios, alphas = [], []
    for L, R, a in s["info"]["splits"]:
        vL, vR = ivp_var_independent(Sigma_np, L), ivp_var_independent(Sigma_np, R)
        alpha_expected = vR / (vL + vR)
        w = s["W"]["HRP"]
        mass_L, mass_R = w[list(L)].sum(), w[list(R)].sum()
        mass_ratios.append(mass_L / (mass_L + mass_R))
        alphas.append(alpha_expected)
    ax.scatter(alphas, mass_ratios, s=18, alpha=0.7, label=str(asof.date()))
ax.plot([0, 1], [0, 1], color="black", ls="--", lw=0.8)
ax.set_xlabel("V_R / (V_L + V_R)")
ax.set_ylabel("mass ratio sum(w_L) / (sum(w_L)+sum(w_R))")
ax.set_title("Split identity check (every node, every snapshot date)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## §4 Backtests

Nine runs, same cost/rebalance/warm-up harness as notebooks 01–06. `HRP(S)`
is the registered run; `GMV(S)`, `EW`, `MDP(S)`, `ERC(S)` are the
reproduction-gate comparators (already backtested in earlier notebooks);
`HRP(LW)`, `HRP[pos](S)`, `HRP[ward](S)` are sensitivities; `IVP` is the
notebook-local inverse-variance reference.

In [ ]:
strategies = {
    "HRP(S)": HierarchicalRiskParity(cov_estimator=ml.sample_cov),
    "GMV(S)": GMV(cov_estimator=ml.sample_cov),
    "EW": EqualWeight(),
    "MDP(S)": MostDiversified(cov_estimator=ml.sample_cov),
    "ERC(S)": EqualRiskContribution(cov_estimator=ml.sample_cov),
    "HRP(LW)": HierarchicalRiskParity(cov_estimator=ml.ledoit_wolf_cov),
    "HRP[pos](S)": HierarchicalRiskParity(bisection="positional"),
    "HRP[ward](S)": HierarchicalRiskParity(linkage="ward"),
    "IVP": InverseVariance(),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    assert wlog.index.equals(rdates), f"{name}: wlog index != rdates"
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

print(f"{len(results)} runs completed.")

### Reproduction gate

`GMV(S)`, `EW`, `MDP(S)`, `ERC(S)` must match notebooks 01/05/06 in
full/train/test Sharpe to 2 dp: GMV 0.62/0.59/0.76, EW 0.74/0.70/0.91,
MDP(S) 0.76/0.70/0.98, ERC(S) 0.76/0.73/0.94. If this doesn't hold, the
harness or estimator wiring changed something and everything downstream is
suspect — so this stops the notebook (`assert`) rather than just printing a
warning.

In [ ]:
rows = []
for name, r in results.items():
    net = r["net"]
    diag = r["diag"]
    turnover = diag["turnover_per_rebalance"]
    win_defs = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        t_window = turnover.loc[t_sel]
        s = summary(net.loc[net_sel], t_window, rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]
summary_table["cost_drag"] = summary_table["ann_turnover"] * ml.COST_BPS / 1e4
print(summary_table.to_string(float_format=F4))

In [ ]:
expected_full_sharpe = {"GMV(S)": 0.62, "EW": 0.74, "MDP(S)": 0.76, "ERC(S)": 0.76}
expected_train_sharpe = {"GMV(S)": 0.59, "EW": 0.70, "MDP(S)": 0.70, "ERC(S)": 0.73}
expected_test_sharpe = {"GMV(S)": 0.76, "EW": 0.91, "MDP(S)": 0.98, "ERC(S)": 0.94}

for name, exp in expected_full_sharpe.items():
    got = round(float(summary_table.loc[(name, "full"), "sharpe"]), 2)
    assert got == exp, f"REPRODUCTION GATE FAILED: {name} full Sharpe {got} != {exp}"
for name, exp in expected_train_sharpe.items():
    got = round(float(summary_table.loc[(name, "train (<= split)"), "sharpe"]), 2)
    assert got == exp, f"REPRODUCTION GATE FAILED: {name} train Sharpe {got} != {exp}"
for name, exp in expected_test_sharpe.items():
    got = round(float(summary_table.loc[(name, "test (> split)"), "sharpe"]), 2)
    assert got == exp, f"REPRODUCTION GATE FAILED: {name} test Sharpe {got} != {exp}"

print("REPRODUCTION GATE: PASSED -- GMV(S), EW, MDP(S), ERC(S) match notebooks 01/05/06 to 2 dp.")

## §5 Mechanism checks, all 208 rebalances × {sample, Ledoit-Wolf} × {tree, positional}

Recomputed fresh with `_hrp_long_only` (not read off the §4 backtests), so
this is an independent check of the closed-form arithmetic itself, not a
tautology. Every tolerance below is asserted at ≤ 1e-12 and every count at
= 208 — HRP has no solver and no iteration, so unlike ERC's coordinate
descent there is no reason to allow any slack at all.

In [ ]:
def quasi_diag_check(Z):
    n = Z.shape[0] + 1
    order = [int(Z[-1, 0]), int(Z[-1, 1])]
    while max(order) >= n:
        new = []
        for c in order:
            if c >= n:
                new += [int(Z[c - n, 0]), int(Z[c - n, 1])]
            else:
                new.append(c)
        order = new
    return np.array(order)


def split_identity_dev(Sigma_np, w, splits):
    dev = 0.0
    for L, R, a in splits:
        vL = ivp_var_independent(Sigma_np, L)
        vR = ivp_var_independent(Sigma_np, R)
        a_expected = vR / (vL + vR)
        dev = max(dev, abs(a - a_expected))
        mL, mR = w[list(L)].sum(), w[list(R)].sum()
        dev = max(dev, abs(mL / (mL + mR) - a_expected))
    return dev


def perm_spread(Sigma_np, bisection, K, seed):
    w0, _ = _hrp_long_only(pd.DataFrame(Sigma_np), "mech5", pd.Timestamp("2020-01-01"), bisection=bisection)
    rng = np.random.default_rng(seed)
    n = Sigma_np.shape[0]
    worst = 0.0
    for _ in range(K):
        p = rng.permutation(n)
        wp, _ = _hrp_long_only(pd.DataFrame(Sigma_np[np.ix_(p, p)]), "mech5", pd.Timestamp("2020-01-01"), bisection=bisection)
        wb = np.empty(n)
        wb[p] = wp
        worst = max(worst, 0.5 * float(np.abs(wb - w0).sum()))
    return worst


ESTS = {"sample": ml.sample_cov, "lw": ml.ledoit_wolf_cov}
hrows = []
pos_spreads_208 = []
cophs = []
root_sides = []

for est_name, est_fn in ESTS.items():
    a = dict(sum_dev=0.0, min_w=np.inf, split_tree=0.0, split_pos=0.0,
             diag_ivp_tree=0.0, diag_ivp_pos=0.0, scale_tree=0.0, scale_pos=0.0,
             perm_tree=0.0, perm_valid=0, link_ok=0, qd_match=0, dist_ok=0, n=0)
    for k, asof in enumerate(rdates):
        _, Sigma = GMV(cov_estimator=est_fn)._estimate(panel, asof)
        Sigma_np = Sigma.to_numpy()
        n_assets = Sigma_np.shape[0]

        w_t, info_t = _hrp_long_only(Sigma, "mech5", asof, bisection="tree")
        w_p, info_p = _hrp_long_only(Sigma, "mech5", asof, bisection="positional")

        a["sum_dev"] = max(a["sum_dev"], abs(w_t.sum() - 1.0), abs(w_p.sum() - 1.0))
        a["min_w"] = min(a["min_w"], float(w_t.min()), float(w_p.min()))
        a["split_tree"] = max(a["split_tree"], split_identity_dev(Sigma_np, w_t, info_t["splits"]))
        a["split_pos"] = max(a["split_pos"], split_identity_dev(Sigma_np, w_p, info_p["splits"]))

        Sigma_diag = pd.DataFrame(np.diag(np.diag(Sigma_np)))
        ivp = (1.0 / np.diag(Sigma_np)) / np.sum(1.0 / np.diag(Sigma_np))
        w_diag_t, _ = _hrp_long_only(Sigma_diag, "mech5", asof, bisection="tree")
        w_diag_p, _ = _hrp_long_only(Sigma_diag, "mech5", asof, bisection="positional")
        a["diag_ivp_tree"] = max(a["diag_ivp_tree"], float(np.max(np.abs(w_diag_t - ivp))))
        a["diag_ivp_pos"] = max(a["diag_ivp_pos"], float(np.max(np.abs(w_diag_p - ivp))))

        w_scale_t, _ = _hrp_long_only(Sigma * 7.3, "mech5", asof, bisection="tree")
        w_scale_p, _ = _hrp_long_only(Sigma * 7.3, "mech5", asof, bisection="positional")
        a["scale_tree"] = max(a["scale_tree"], float(np.max(np.abs(w_scale_t - w_t))))
        a["scale_pos"] = max(a["scale_pos"], float(np.max(np.abs(w_scale_p - w_p))))

        a["perm_tree"] = max(a["perm_tree"], perm_spread(Sigma_np, "tree", 5, seed=k))
        a["perm_valid"] += int(np.array_equal(np.sort(info_t["order"]), np.arange(n_assets)))

        Z = info_t["Z"]
        a["link_ok"] += int(is_valid_linkage(Z) and is_monotonic(Z))
        a["qd_match"] += int(np.array_equal(quasi_diag_check(Z), info_t["order"]))

        sigma_np = np.sqrt(np.diag(Sigma_np))
        C = np.clip(0.5 * ((Sigma_np / np.outer(sigma_np, sigma_np)) + (Sigma_np / np.outer(sigma_np, sigma_np)).T), -1.0, 1.0)
        np.fill_diagonal(C, 1.0)
        D = np.sqrt(0.5 * (1.0 - C))
        np.fill_diagonal(D, 0.0)
        a["dist_ok"] += int(np.array_equal(D, D.T) and np.all(np.diag(D) == 0.0) and D.min() >= 0.0 and D.max() <= 1.0)
        a["n"] += 1

        if est_name == "sample":
            pos_spreads_208.append(perm_spread(Sigma_np, "positional", 20, seed=1000 + k))
            y = pdist(D, metric="euclidean")
            cophs.append(float(cophenet(Z, y)[0]))
            root_node = sch.to_tree(Z)
            root_L = frozenset(root_node.get_left().pre_order())
            iU_here = list(Sigma.columns).index("UUP")
            uup_side = root_L if iU_here in root_L else frozenset(root_node.get_right().pre_order())
            root_sides.append(uup_side)

    hrows.append(pd.Series(a, name=est_name))

H = pd.DataFrame(hrows)
print(H.to_string(float_format=E3))

tol = 1e-12
for _, row in H.iterrows():
    assert row["sum_dev"] <= tol
    assert row["min_w"] > 0
    assert row["split_tree"] <= tol
    assert row["split_pos"] <= tol
    assert row["diag_ivp_tree"] <= tol
    assert row["diag_ivp_pos"] <= tol
    assert row["scale_tree"] <= tol
    assert row["scale_pos"] <= tol
    assert row["perm_tree"] <= tol
    assert row["perm_valid"] == 208
    assert row["link_ok"] == 208
    assert row["qd_match"] == 208
    assert row["dist_ok"] == 208

print("Mechanism checks: PASSED at all 208 rebalances, both estimators, both bisections.")

ps = np.array(pos_spreads_208)
n_changes = sum(1 for i in range(1, len(root_sides)) if root_sides[i] != root_sides[i - 1])
print(f"positional permutation spread (K=20, sample Sigma): max={E3(ps.max())}  "
      f"n>1e-12={(ps > 1e-12).sum()}  n>1%={(ps > 0.01).sum()}")
print(f"median cophenetic correlation (sample Sigma, tree): {F4(float(np.median(cophs)))}")
print(f"root-split membership changes across 208 rebalances (HRP(S), tree): {n_changes}")

## §6 Hypothesis tests

### 6.1 — H1a / H1b (the UUP corner, from target weights)

In [ ]:
def count_rule(holds_2022, frac_full, frac_test):
    if holds_2022 and frac_full > 0.5 and frac_test > 0.5:
        return "SUPPORTED"
    if holds_2022 and frac_full > 0.5:
        return "NOT ROBUST"
    return "NOT SUPPORTED"


w_uup_hrp_ts = results["HRP(S)"]["wlog"]["UUP"]
w_uup_gmv_ts = results["GMV(S)"]["wlog"]["UUP"]
w_uup_mdp_ts = results["MDP(S)"]["wlog"]["UUP"]
w_uup_erc_ts = results["ERC(S)"]["wlog"]["UUP"]

lt_gmv = w_uup_hrp_ts < w_uup_gmv_ts
lt_mdp = w_uup_hrp_ts < w_uup_mdp_ts
h1a_full = lt_gmv & lt_mdp
n_full = len(h1a_full)
test_mask = h1a_full.index > split_ts
h1a_test = h1a_full.loc[test_mask]
n_test = int(test_mask.sum())
frac_full_h1a = float(h1a_full.mean())
frac_test_h1a = float(h1a_test.mean())

uup_2022 = {
    "HRP(S)": float(w_uup_hrp_ts.loc[asof_2022]),
    "GMV(S)": float(w_uup_gmv_ts.loc[asof_2022]),
    "MDP(S)": float(w_uup_mdp_ts.loc[asof_2022]),
    "ERC(S)": float(w_uup_erc_ts.loc[asof_2022]),
}
h1a_holds_2022 = uup_2022["HRP(S)"] < uup_2022["GMV(S)"] and uup_2022["HRP(S)"] < uup_2022["MDP(S)"]
h1a_verdict = count_rule(h1a_holds_2022, frac_full_h1a, frac_test_h1a)

print(f"H1a (HRP<GMV and HRP<MDP): full {int(h1a_full.sum())}/{n_full} ({100*frac_full_h1a:.1f}%), "
      f"test {int(h1a_test.sum())}/{n_test} ({100*frac_test_h1a:.1f}%)  -> {h1a_verdict}")
print(f"2022-12-31 w_UUP: HRP={uup_2022['HRP(S)']:.4f}  GMV={uup_2022['GMV(S)']:.4f}  "
      f"MDP={uup_2022['MDP(S)']:.4f}  ERC={uup_2022['ERC(S)']:.4f}")

h1b_full = w_uup_hrp_ts < w_uup_erc_ts
h1b_test = h1b_full.loc[test_mask]
print(f"\nH1b (descriptive, HRP<ERC): full {int(h1b_full.sum())}/{n_full} ({100*h1b_full.mean():.1f}%), "
      f"test {int(h1b_test.sum())}/{n_test} ({100*h1b_test.mean():.1f}%)")

### 6.2 — H2a / H2b (root singleton, correlation blindness)

Recomputed fresh (sample Σ) at every rebalance, including the Σ_cf
counterfactual — this is not read off the §4 backtests.

In [ ]:
h2_rows = []
for asof in rdates:
    _, Sigma = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof)
    cols = list(Sigma.columns)
    Sigma_np = Sigma.to_numpy()
    iU = cols.index("UUP")

    w_hrp, info_hrp = _hrp_long_only(Sigma, "h2", asof, bisection="tree")
    w_gmv = _min_variance_long_only(Sigma)
    w_mdp, _ = _mdp_long_only(Sigma, "h2", asof)
    w_erc, _ = _erc_long_only(Sigma, "h2", asof)

    Sigma_cf_np = sigma_cf_zero_corr(Sigma_np, iU)
    Sigma_cf = pd.DataFrame(Sigma_cf_np, index=cols, columns=cols)
    w_hrp_cf, info_hrp_cf = _hrp_long_only(Sigma_cf, "h2cf", asof, bisection="tree")
    w_gmv_cf = _min_variance_long_only(Sigma_cf)
    w_mdp_cf, _ = _mdp_long_only(Sigma_cf, "h2cf", asof)
    w_erc_cf, _ = _erc_long_only(Sigma_cf, "h2cf", asof)

    root_s = root_singleton(info_hrp["Z"], iU)
    root_s_cf = root_singleton(info_hrp_cf["Z"], iU)

    h2_rows.append({
        "asof": asof,
        "root_singleton": root_s, "root_singleton_cf": root_s_cf,
        "dw_HRP": abs(w_hrp[iU] - w_hrp_cf[iU]),
        "dw_GMV": abs(w_gmv[iU] - w_gmv_cf[iU]),
        "dw_MDP": abs(w_mdp[iU] - w_mdp_cf[iU]),
        "dw_ERC": abs(w_erc[iU] - w_erc_cf[iU]),
    })

h2_table = pd.DataFrame(h2_rows).set_index("asof")
print("h2_table computed:", h2_table.shape)

In [ ]:
n_h2a = int(h2_table["root_singleton"].sum())
n_both_singleton = int((h2_table["root_singleton"] & h2_table["root_singleton_cf"]).sum())
print(f"H2a (descriptive): UUP root singleton at {n_h2a}/208 rebalances")
print(f"both-singleton count (Sigma and Sigma_cf): {n_both_singleton}/208")

test_mask_h2 = h2_table.index > split_ts
h2b_verdicts = {}
for comp in ["GMV", "MDP", "ERC"]:
    holds = h2_table["dw_HRP"] < h2_table[f"dw_{comp}"]
    frac_full = float(holds.mean())
    frac_test = float(holds.loc[test_mask_h2].mean())
    holds_2022 = bool(holds.loc[asof_2022]) if asof_2022 in holds.index else bool(holds.iloc[-1])
    verdict = count_rule(holds_2022, frac_full, frac_test)
    h2b_verdicts[comp] = dict(frac_full=frac_full, frac_test=frac_test, holds_2022=holds_2022, verdict=verdict)
    print(f"H2b vs {comp}: full {int(holds.sum())}/208 ({100*frac_full:.1f}%), "
          f"test {int(holds.loc[test_mask_h2].sum())}/{int(test_mask_h2.sum())} ({100*frac_test:.1f}%), "
          f"2022-12-31 holds={holds_2022}  -> {verdict}")

### 6.3 — H3 (effective N, from target weights of the §4 backtests)

In [ ]:
effN_hrp_ts = 1.0 / (results["HRP(S)"]["wlog"] ** 2).sum(axis=1)
effN_gmv_ts = 1.0 / (results["GMV(S)"]["wlog"] ** 2).sum(axis=1)
effN_mdp_ts = 1.0 / (results["MDP(S)"]["wlog"] ** 2).sum(axis=1)
effN_erc_ts = 1.0 / (results["ERC(S)"]["wlog"] ** 2).sum(axis=1)

h3_specs = [
    ("GMV", effN_gmv_ts, lambda h, c: h > c),
    ("MDP", effN_mdp_ts, lambda h, c: h > c),
    ("ERC", effN_erc_ts, lambda h, c: h < c),
]
h3_verdicts = {}
for comp, comp_ts, op in h3_specs:
    holds = op(effN_hrp_ts, comp_ts)
    frac_full = float(holds.mean())
    frac_test = float(holds.loc[test_mask].mean())
    holds_2022 = bool(holds.loc[asof_2022])
    verdict = count_rule(holds_2022, frac_full, frac_test)
    h3_verdicts[comp] = dict(frac_full=frac_full, frac_test=frac_test, holds_2022=holds_2022, verdict=verdict)
    direction = ">" if op(1, 0) else "<"
    print(f"H3 effN(HRP) {direction} effN({comp}): full {int(holds.sum())}/208 ({100*frac_full:.1f}%), "
          f"test {int(holds.loc[test_mask].sum())}/{int(test_mask.sum())} ({100*frac_test:.1f}%), "
          f"2022-12-31 holds={holds_2022}  -> {verdict}")

print(f"\nmedian effN: HRP(S)={float(effN_hrp_ts.median()):.4f}  GMV(S)={float(effN_gmv_ts.median()):.4f}  "
      f"MDP(S)={float(effN_mdp_ts.median()):.4f}  ERC(S)={float(effN_erc_ts.median()):.4f}")
print(f"2022-12-31 effN: HRP={float(effN_hrp_ts.loc[asof_2022]):.4f}  GMV={float(effN_gmv_ts.loc[asof_2022]):.4f}  "
      f"MDP={float(effN_mdp_ts.loc[asof_2022]):.4f}  ERC={float(effN_erc_ts.loc[asof_2022]):.4f}")

### 6.4 — Paired daily-difference table (H4, H5, H6)

In [ ]:
pairs = [
    ("HRP(S)", "GMV(S)"),
    ("HRP(S)", "MDP(S)"),
    ("HRP(S)", "ERC(S)"),
    ("HRP(S)", "EW"),
    ("HRP(LW)", "HRP(S)"),
    ("HRP(S)", "IVP"),
    ("HRP[pos](S)", "HRP(S)"),
    ("HRP[ward](S)", "HRP(S)"),
]

paired_rows = []
paired_series = {}
for a_name, b_name in pairs:
    net_a = results[a_name]["net"]
    net_b = results[b_name]["net"]
    idx = net_a.index.intersection(net_b.index)
    diff = net_a.loc[idx] - net_b.loc[idx]
    paired_series[(a_name, b_name)] = diff
    win_defs = {
        "full": diff.index >= diff.index.min(),
        "train (<= split)": diff.index <= split_ts,
        "test (> split)": diff.index > split_ts,
    }
    for window_name, sel in win_defs.items():
        d = diff.loc[sel]
        ann_mean = float(d.mean() * ml.TRADING_DAYS)
        te = float(d.std(ddof=1) * np.sqrt(ml.TRADING_DAYS))
        t_stat = float(d.mean() / (d.std(ddof=1) / np.sqrt(len(d))))
        cost_drag_a = float(summary_table.loc[(a_name, window_name), "cost_drag"])
        cost_drag_b = float(summary_table.loc[(b_name, window_name), "cost_drag"])
        cost_drag_diff = cost_drag_a - cost_drag_b
        gross_diff = ann_mean + cost_drag_diff
        paired_rows.append({
            "pair": f"{a_name} - {b_name}", "window": window_name,
            "ann_mean_diff": ann_mean, "TE": te, "t": t_stat,
            "cost_drag_diff": cost_drag_diff, "gross_of_costs_mean_diff": gross_diff,
        })

paired_table = pd.DataFrame(paired_rows).set_index(["pair", "window"])
print(paired_table.to_string(float_format=F4))

### 6.5 — CAPM (context only, not pre-registered) and the diff-regression Δα used for H4/H6

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_rows = []
for name in strategies:
    net = results[name]["net"]
    idx = net.index.intersection(rf_daily.index).intersection(MKT_simple.index)
    strat_excess = net.loc[idx] - rf_daily.loc[idx]
    mkt_aligned = MKT_simple.loc[idx]
    fit = ols_with_intercept(strat_excess, mkt_aligned.to_frame("MKT"))
    capm_rows.append({
        "strategy": name,
        "alpha_ann": float(fit["coef"]["alpha"] * ml.TRADING_DAYS),
        "t_alpha": float(fit["tstat"]["alpha"]),
        "beta": float(fit["coef"]["MKT"]),
        "r2": fit["r2"], "n": fit["n"],
    })
capm_table = pd.DataFrame(capm_rows).set_index("strategy")
print(capm_table.to_string(float_format=F4))

In [ ]:
diff_capm_rows = []
for a_name, b_name in pairs:
    diff = paired_series[(a_name, b_name)]
    idx = diff.index.intersection(MKT_simple.index)
    fit = ols_with_intercept(diff.loc[idx], MKT_simple.loc[idx].to_frame("MKT"))
    diff_capm_rows.append({
        "pair": f"{a_name} - {b_name}",
        "alpha_diff_ann": float(fit["coef"]["alpha"] * ml.TRADING_DAYS),
        "t_alpha_diff": float(fit["tstat"]["alpha"]),
        "beta_diff": float(fit["coef"]["MKT"]),
        "n": fit["n"],
    })
diff_capm_table = pd.DataFrame(diff_capm_rows).set_index("pair")
print(diff_capm_table.to_string(float_format=F4))

In [ ]:
decomp_results = {}
for a_name, b_name in pairs:
    alpha_a = float(capm_table.loc[a_name, "alpha_ann"]); beta_a = float(capm_table.loc[a_name, "beta"])
    alpha_b = float(capm_table.loc[b_name, "alpha_ann"]); beta_b = float(capm_table.loc[b_name, "beta"])

    net_a = results[a_name]["net"]
    idx = net_a.index.intersection(rf_daily.index).intersection(MKT_simple.index)
    mkt_ann_mean = float(MKT_simple.loc[idx].mean() * ml.TRADING_DAYS)

    d_alpha = alpha_a - alpha_b
    d_beta = beta_a - beta_b
    beta_part = d_beta * mkt_ann_mean
    decomposed = d_alpha + beta_part

    realized_diff_full = float(paired_table.loc[(f"{a_name} - {b_name}", "full"), "ann_mean_diff"])
    identity_check = realized_diff_full - decomposed

    pair_key = f"{a_name} - {b_name}"
    t_alpha_diff = float(diff_capm_table.loc[pair_key, "t_alpha_diff"])

    print(f"=== {pair_key} (full window) ===")
    print(f"  t_alpha[{a_name}]={float(capm_table.loc[a_name, 't_alpha']):.2f}  t_alpha[{b_name}]={float(capm_table.loc[b_name, 't_alpha']):.2f}")
    print(f"  d_alpha (ann)         = {d_alpha:.4%}   (direct diff-regression: {float(diff_capm_table.loc[pair_key, 'alpha_diff_ann']):.4%}, t={t_alpha_diff:.2f})")
    print(f"  d_beta                = {d_beta:.4f}")
    print(f"  ann mean MKT excess   = {mkt_ann_mean:.4%}")
    print(f"  beta_part = d_beta * mkt_ann_mean = {beta_part:.4%}")
    print(f"  d_alpha + beta_part   = {decomposed:.4%}")
    print(f"  realized ann_mean_diff (full) = {realized_diff_full:.4%}")
    print(f"  identity check (0 by OLS construction) = {identity_check:.4%}")
    print()

    decomp_results[pair_key] = dict(d_alpha=d_alpha, beta_part=beta_part, d_beta=d_beta,
                                     mkt_ann_mean=mkt_ann_mean, realized_diff_full=realized_diff_full)

### 6.6 — H7 (forecast bias: realized vs. ex-ante vol, HRP vs. GMV)

In [ ]:
h7_methods = ["HRP(S)", "GMV(S)", "MDP(S)", "ERC(S)", "IVP", "EW"]

sigma_cache = {}
for asof in rdates:
    _, Sigma = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof)
    sigma_cache[asof] = Sigma

boundaries = list(rdates) + [simple_returns.index.max() + pd.Timedelta(days=1)]
q_rows = []
n_dropped = 0
for k, asof in enumerate(rdates):
    start, end = boundaries[k], boundaries[k + 1]
    Sigma = sigma_cache[asof]
    Sigma_np = Sigma.to_numpy()
    for m in h7_methods:
        net = results[m]["net"]
        seg_idx = net.index[(net.index >= start) & (net.index < end)]
        if len(seg_idx) <= 1:
            n_dropped += 1
            continue
        seg_idx_excl_first = seg_idx[1:]
        if len(seg_idx_excl_first) < 10:
            n_dropped += 1
            continue
        realized_vol = float(net.loc[seg_idx_excl_first].std(ddof=1) * np.sqrt(ml.TRADING_DAYS))

        w = results[m]["wlog"].loc[asof].reindex(Sigma.columns).to_numpy()
        exante_vol = port_vol(w, Sigma_np)
        q = float(np.log(realized_vol / exante_vol))
        q_rows.append({"asof": asof, "method": m, "q": q, "exante_vol": exante_vol, "realized_vol": realized_vol})

q_table = pd.DataFrame(q_rows)
print(f"segments dropped (< 10 remaining days after excluding cost-booking day): {n_dropped}")

In [ ]:
q_pivot = q_table.pivot(index="asof", columns="method", values="q")
test_mask_q = q_pivot.index > split_ts

print("mean q by method (full / test):")
q_means = pd.DataFrame({
    "full": q_pivot.mean(),
    "test": q_pivot.loc[test_mask_q].mean(),
})
print(q_means.to_string(float_format=F4))

d_h7 = (q_pivot["HRP(S)"] - q_pivot["GMV(S)"]).dropna()
d_full = d_h7
d_test = d_h7.loc[d_h7.index > split_ts]

t_full = float(d_full.mean() / (d_full.std(ddof=1) / np.sqrt(len(d_full))))
t_test = float(d_test.mean() / (d_test.std(ddof=1) / np.sqrt(len(d_test)))) if len(d_test) > 1 else float("nan")

if t_full <= -2 and float(d_test.mean()) < 0:
    h7_verdict = "SUPPORTED"
elif t_full <= -2:
    h7_verdict = "NOT ROBUST"
else:
    h7_verdict = "NOT SUPPORTED"

print(f"\nH7: d = q_HRP - q_GMV  full mean={float(d_full.mean()):.4f} t={t_full:.2f} n={len(d_full)}   "
      f"test mean={float(d_test.mean()):.4f} t={t_test:.2f} n={len(d_test)}  -> {h7_verdict}")

Reading note: every method's mean q is negative (realized vol below the 252-day ex-ante vol), so H7 is a RELATIVE statement — which method's forecast is least conservative — not evidence that any method's ex-ante vol is too low.

### 6.7 — VERDICT TABLE (mechanical application of the §2 rules)

In [ ]:
def null_verdict(t_abs):
    if t_abs < 1.5:
        return "NULL HOLDS"
    if t_abs < 2.0:
        return "NULL HOLDS NARROWLY"
    return "NULL REJECTED"


verdict_rows = []

verdict_rows.append({
    "item": "H1a (w_UUP HRP<GMV and <MDP)",
    "statistic": f"full {int(h1a_full.sum())}/{n_full}, test {int(h1a_test.sum())}/{n_test}, "
                 f"2022-12-31 holds={h1a_holds_2022}",
    "rule": "count rule", "verdict": h1a_verdict,
})

for comp in ["GMV", "MDP", "ERC"]:
    v = h2b_verdicts[comp]
    verdict_rows.append({
        "item": f"H2b (|dw_UUP_cf| HRP < {comp})",
        "statistic": f"full {100*v['frac_full']:.1f}%, test {100*v['frac_test']:.1f}%, 2022 holds={v['holds_2022']}",
        "rule": "count rule", "verdict": v["verdict"],
    })

for comp in ["GMV", "MDP", "ERC"]:
    v = h3_verdicts[comp]
    verdict_rows.append({
        "item": f"H3 (effN HRP vs {comp})",
        "statistic": f"full {100*v['frac_full']:.1f}%, test {100*v['frac_test']:.1f}%, 2022 holds={v['holds_2022']}",
        "rule": "count rule", "verdict": v["verdict"],
    })

for comp in ["GMV(S)", "MDP(S)", "ERC(S)"]:
    pair_key = f"HRP(S) - {comp}"
    t = float(diff_capm_table.loc[pair_key, "t_alpha_diff"])
    verdict_rows.append({
        "item": f"H4 (Delta-alpha, HRP(S)-{comp})",
        "statistic": f"t={t:.2f}", "rule": "|t|<2 null", "verdict": null_verdict(abs(t)),
    })

d5 = decomp_results["HRP(S) - EW"]
t_full_ew = float(paired_table.loc[("HRP(S) - EW", "full"), "t"])
mean_test_ew = float(paired_table.loc[("HRP(S) - EW", "test (> split)"), "ann_mean_diff"])
h5_cond = t_full_ew <= -2 and mean_test_ew < 0 and abs(d5["beta_part"]) > abs(d5["d_alpha"])
h5_verdict = "SUPPORTED" if h5_cond else "NOT SUPPORTED"
verdict_rows.append({
    "item": "H5 (HRP-EW < 0, beta-driven)",
    "statistic": f"t_full={t_full_ew:.2f}, mean_test={mean_test_ew:.4f}, "
                 f"|beta_part|={abs(d5['beta_part']):.4f} vs |d_alpha|={abs(d5['d_alpha']):.4f}",
    "rule": "H5 rule", "verdict": h5_verdict,
})

t_full_lw = float(paired_table.loc[("HRP(LW) - HRP(S)", "full"), "t"])
t_test_lw = float(paired_table.loc[("HRP(LW) - HRP(S)", "test (> split)"), "t"])
t_h6_lw = max(abs(t_full_lw), abs(t_test_lw))
verdict_rows.append({
    "item": "H6 (HRP(LW)-HRP(S), total)",
    "statistic": f"t_full={t_full_lw:.2f}, t_test={t_test_lw:.2f}",
    "rule": "|t|<2 null (larger of full/test)", "verdict": null_verdict(t_h6_lw),
})

for pair_key, label in [
    ("HRP(S) - IVP", "H6 (Delta-alpha, HRP(S)-IVP)"),
    ("HRP[pos](S) - HRP(S)", "H6 (Delta-alpha, HRP[pos](S)-HRP(S))"),
    ("HRP[ward](S) - HRP(S)", "H6 (Delta-alpha, HRP[ward](S)-HRP(S))"),
]:
    t = float(diff_capm_table.loc[pair_key, "t_alpha_diff"])
    verdict_rows.append({
        "item": label, "statistic": f"t={t:.2f}", "rule": "|t|<2 null", "verdict": null_verdict(abs(t)),
    })

verdict_rows.append({
    "item": "H7 (forecast bias, q_HRP-q_GMV < 0)",
    "statistic": f"t_full={t_full:.2f}, mean_test={float(d_test.mean()):.4f}",
    "rule": "H7 rule", "verdict": h7_verdict,
})

verdict_table = pd.DataFrame(verdict_rows)
print(verdict_table.to_string(index=False))

print("\n--- descriptive rows ---")
print(f"H1b (descriptive, HRP<ERC): full {int(h1b_full.sum())}/{n_full}, test {int(h1b_test.sum())}/{n_test}")
print(f"H2a (descriptive, UUP root singleton): {n_h2a}/208")
print(f"both-singleton count (Sigma and Sigma_cf): {n_both_singleton}/208")

## §7 Figures

### F3 — group-weight stackplot

Which asset groups carry the target weight over time under HRP(S) vs.
ERC(S) vs. MDP(S) vs. GMV(S)?

In [ ]:
group_order = list(ml.ASSET_GROUPS.keys())
group_colors = plt.cm.tab10(np.linspace(0, 1, len(group_order)))

def weights_by_group(wlog):
    grouped = wlog.T.groupby(ml.GROUP_OF).sum().T
    return grouped.reindex(columns=group_order).fillna(0.0)

hrp_group = weights_by_group(results["HRP(S)"]["wlog"])
erc_group = weights_by_group(results["ERC(S)"]["wlog"])
mdp_group = weights_by_group(results["MDP(S)"]["wlog"])
gmv_group = weights_by_group(results["GMV(S)"]["wlog"])

fig, axes = plt.subplots(4, 1, figsize=(9, 13), sharex=True)
for ax, grouped, title in zip(
    axes, [hrp_group, erc_group, mdp_group, gmv_group], ["HRP(S)", "ERC(S)", "MDP(S)", "GMV(S)"]
):
    ax.stackplot(grouped.index, grouped.T.to_numpy(), labels=group_order, colors=group_colors)
    ax.axvline(split_ts, color="black", ls=":", lw=1)
    ax.set_ylim(0, 1)
    ax.set_title(f"{title} -- target weights by asset group")
axes[-1].set_xlabel("rebalance date")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
plt.tight_layout(); plt.show()

print("Mean target weight by asset group:")
group_means = pd.DataFrame({
    "HRP(S)": hrp_group.mean(), "ERC(S)": erc_group.mean(),
    "MDP(S)": mdp_group.mean(), "GMV(S)": gmv_group.mean(),
})
print(group_means.to_string(float_format=F4))

### F4 — effN, max weight, and UUP weight over time

Is the effN/UUP contrast between HRP, ERC, MDP and GMV a snapshot artifact
or does it hold across the full 208-rebalance history?

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9.5, 11), sharex=True)

effN_ew_ts = 1.0 / (results["EW"]["wlog"] ** 2).sum(axis=1)
axes[0].plot(effN_hrp_ts.index, effN_hrp_ts.to_numpy(), label="HRP(S)", color="#8e44ad", lw=1.2)
axes[0].plot(effN_erc_ts.index, effN_erc_ts.to_numpy(), label="ERC(S)", color="#d4a017", lw=1.2)
axes[0].plot(effN_mdp_ts.index, effN_mdp_ts.to_numpy(), label="MDP(S)", color=ml.FAMILY_COLORS["Risk-based"], lw=1.2)
axes[0].plot(effN_gmv_ts.index, effN_gmv_ts.to_numpy(), label="GMV(S)", color="#555555", lw=1.2)
axes[0].plot(effN_ew_ts.index, effN_ew_ts.to_numpy(), label="EW", color="#999999", lw=0.8, ls="--")
axes[0].axvline(split_ts, color="black", ls=":", lw=1)
axes[0].set_ylabel("effective N")
axes[0].legend(fontsize=8)

maxw_hrp_ts = results["HRP(S)"]["wlog"].max(axis=1)
maxw_erc_ts = results["ERC(S)"]["wlog"].max(axis=1)
maxw_mdp_ts = results["MDP(S)"]["wlog"].max(axis=1)
maxw_gmv_ts = results["GMV(S)"]["wlog"].max(axis=1)
axes[1].plot(maxw_hrp_ts.index, maxw_hrp_ts.to_numpy() * 100, label="HRP(S)", color="#8e44ad", lw=1.2)
axes[1].plot(maxw_erc_ts.index, maxw_erc_ts.to_numpy() * 100, label="ERC(S)", color="#d4a017", lw=1.2)
axes[1].plot(maxw_mdp_ts.index, maxw_mdp_ts.to_numpy() * 100, label="MDP(S)", color=ml.FAMILY_COLORS["Risk-based"], lw=1.2)
axes[1].plot(maxw_gmv_ts.index, maxw_gmv_ts.to_numpy() * 100, label="GMV(S)", color="#555555", lw=1.2)
axes[1].axvline(split_ts, color="black", ls=":", lw=1)
axes[1].set_ylabel("max weight (%)")
axes[1].legend(fontsize=8)

axes[2].plot(w_uup_hrp_ts.index, w_uup_hrp_ts.to_numpy() * 100, label="HRP(S)", color="#8e44ad", lw=1.2)
axes[2].plot(w_uup_erc_ts.index, w_uup_erc_ts.to_numpy() * 100, label="ERC(S)", color="#d4a017", lw=1.2)
axes[2].plot(w_uup_mdp_ts.index, w_uup_mdp_ts.to_numpy() * 100, label="MDP(S)", color=ml.FAMILY_COLORS["Risk-based"], lw=1.2)
axes[2].plot(w_uup_gmv_ts.index, w_uup_gmv_ts.to_numpy() * 100, label="GMV(S)", color="#555555", lw=1.2)
axes[2].axvline(split_ts, color="black", ls=":", lw=1)
axes[2].set_ylabel("UUP weight (%)")
axes[2].set_xlabel("rebalance date")
axes[2].legend(fontsize=8)

fig.suptitle("HRP(S) vs ERC(S) vs MDP(S) vs GMV(S): effN, max weight, UUP weight")
plt.tight_layout(); plt.show()

### F5 — cumulative paired differences

For the Δα-registered pairs, is the beta-adjusted difference a persistent
drift or does it track the raw (total) difference? For HRP-EW and
HRP(LW)-HRP(S), is the total difference a persistent drift or a noisy path?

In [ ]:
alpha_pairs = [
    ("HRP(S)", "GMV(S)"), ("HRP(S)", "MDP(S)"), ("HRP(S)", "ERC(S)"),
    ("HRP(S)", "IVP"), ("HRP[pos](S)", "HRP(S)"), ("HRP[ward](S)", "HRP(S)"),
]
other_pairs = [("HRP(S)", "EW"), ("HRP(LW)", "HRP(S)")]

fig, axes = plt.subplots(2, 1, figsize=(9.5, 9), sharex=True)

ax = axes[0]
for a_name, b_name in alpha_pairs:
    pair_key = f"{a_name} - {b_name}"
    diff = paired_series[(a_name, b_name)]
    beta_hat = float(diff_capm_table.loc[pair_key, "beta_diff"])
    idx = diff.index.intersection(MKT_simple.index)
    adj = (diff.loc[idx] - beta_hat * MKT_simple.loc[idx]).cumsum() * 100
    total = diff.cumsum() * 100
    line, = ax.plot(adj.index, adj.to_numpy(), lw=1.1, label=pair_key)
    ax.plot(total.index, total.to_numpy(), lw=0.8, ls="--", color=line.get_color())
ax.axvline(split_ts, color="black", ls=":", lw=1)
ax.axhline(0, color="black", lw=0.6)
ax.set_ylabel("cumulative diff (%)")
ax.set_title("(a) Delta-alpha registered pairs: solid = beta-adjusted, dashed = total")
ax.legend(fontsize=7, ncol=2)

ax = axes[1]
for a_name, b_name in other_pairs:
    diff = paired_series[(a_name, b_name)]
    cum = diff.cumsum() * 100
    ax.plot(cum.index, cum.to_numpy(), lw=1.1, label=f"{a_name} - {b_name}")
ax.axvline(split_ts, color="black", ls=":", lw=1, label="train/test split")
ax.axhline(0, color="black", lw=0.6)
ax.set_ylabel("cumulative diff (%)")
ax.set_xlabel("date")
ax.set_title("(b) HRP-EW and HRP(LW)-HRP(S): total")
ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

### F6 — forecast bias over time

Is the HRP-vs-GMV ex-ante/realized vol gap concentrated in a particular
period, or roughly constant across the sample?

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5))
q_hrp = q_pivot["HRP(S)"].dropna()
q_gmv = q_pivot["GMV(S)"].dropna()
ax.plot(q_hrp.index, q_hrp.to_numpy(), label="q_HRP", color="#8e44ad", lw=1.1)
ax.plot(q_gmv.index, q_gmv.to_numpy(), label="q_GMV", color="#555555", lw=1.1)
ax.axhline(0, color="black", lw=0.6)
ax.axvline(split_ts, color="black", ls=":", lw=1, label="train/test split")
ax.set_ylabel("q = log(realized vol / ex-ante vol)")
ax.set_xlabel("rebalance date")
ax.set_title("Forecast bias per holding segment: HRP vs GMV")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### F7 — root-split decomposition: why the corner comes back

Whenever UUP is a root singleton, HRP decides its weight in ONE comparison: σ²_UUP against V_rest, the variance of the inverse-variance portfolio of the other 12 assets (w_UUP = V_rest/(σ²_UUP &#43; V_rest)). How do those two quantities move over the 208 rebalances? Descriptive only — not a pre-registered test.

In [ ]:
f7_rows = []
for asof in rdates:
    Sigma = sigma_cache[asof]
    cols = list(Sigma.columns)
    iU = cols.index("UUP")
    Sigma_np = Sigma.to_numpy()
    rest_idx = [j for j in range(len(cols)) if j != iU]

    sigma_uup = float(np.sqrt(Sigma_np[iU, iU]))
    V_rest = ivp_var_independent(Sigma_np, rest_idx)
    sqrt_V_rest = float(np.sqrt(V_rest))
    identity = V_rest / (sigma_uup ** 2 + V_rest)

    sub = Sigma_np[np.ix_(rest_idx, rest_idx)]
    sub_sigma = np.sqrt(np.diag(sub))
    sub_C = sub / np.outer(sub_sigma, sub_sigma)
    n_rest = len(rest_idx)
    rho_bar_rest = float((sub_C.sum() - n_rest) / (n_rest * (n_rest - 1)))

    f7_rows.append({
        "asof": asof,
        "root_singleton": bool(h2_table.loc[asof, "root_singleton"]),
        "sigma_UUP": sigma_uup,
        "sqrt_V_rest": sqrt_V_rest,
        "rho_bar_rest": rho_bar_rest,
        "identity": identity,
        "w_UUP_HRP": float(results["HRP(S)"]["wlog"].loc[asof, "UUP"]),
        "w_UUP_GMV": float(results["GMV(S)"]["wlog"].loc[asof, "UUP"]),
        "w_UUP_ERC": float(results["ERC(S)"]["wlog"].loc[asof, "UUP"]),
    })
f7_table = pd.DataFrame(f7_rows).set_index("asof")

root_mask = f7_table["root_singleton"]
max_dev = float((f7_table.loc[root_mask, "w_UUP_HRP"] - f7_table.loc[root_mask, "identity"]).abs().max())
assert max_dev <= 1e-12, f"root-singleton identity check failed: max_dev={max_dev:.3e}"
print(f"max |w_UUP(HRP) - identity| over root-singleton rebalances: {max_dev:.3e}")

dates_idx = f7_table.index
non_singleton_dates = dates_idx[~f7_table["root_singleton"].to_numpy()]
diffs = dates_idx.to_series().diff().dropna()
half_width = diffs.median() / 2 if len(diffs) else pd.Timedelta(days=15)

def shade_non_singleton(ax):
    for d in non_singleton_dates:
        ax.axvspan(d - half_width, d + half_width, color="#dddddd", alpha=0.6, lw=0, zorder=0)

fig, axes = plt.subplots(3, 1, figsize=(9.5, 11), sharex=True)

ax = axes[0]
shade_non_singleton(ax)
ax.plot(f7_table.index, f7_table["sigma_UUP"], label="sigma_UUP", color="#c0392b", lw=1.1)
ax.plot(f7_table.index, f7_table["sqrt_V_rest"], label="sqrt(V_rest)", color="#555555", lw=1.1)
ax.axvline(split_ts, color="black", ls=":", lw=1, label="train/test split")
ax.set_ylabel("annualized vol")
ax.set_title("F7(a): sigma_UUP vs sqrt(V_rest)", fontsize=9)
ax.legend(fontsize=8)

ax = axes[1]
shade_non_singleton(ax)
ax.plot(f7_table.index, f7_table["w_UUP_HRP"], label="w_UUP HRP(S)", color="#8e44ad", lw=1.3)
ax.plot(f7_table.index, f7_table["identity"], label="identity V_rest/(sigma_UUP^2+V_rest)",
        color="#8e44ad", lw=1.0, ls="--")
ax.plot(f7_table.index, f7_table["w_UUP_GMV"], label="w_UUP GMV(S)",
        color=ml.FAMILY_COLORS["Risk-based"], lw=0.7)
ax.plot(f7_table.index, f7_table["w_UUP_ERC"], label="w_UUP ERC(S)", color="#d4a017", lw=0.7)
ax.axvline(split_ts, color="black", ls=":", lw=1)
ax.set_ylabel("w_UUP")
ax.set_title("F7(b): HRP w_UUP vs the root-singleton identity", fontsize=9)
ax.legend(fontsize=7, ncol=2)

ax = axes[2]
shade_non_singleton(ax)
ax.plot(f7_table.index, f7_table["rho_bar_rest"], color="#2c3e50", lw=1.1)
ax.axhline(0, color="black", lw=0.5)
ax.axvline(split_ts, color="black", ls=":", lw=1)
ax.set_ylabel("mean pairwise corr\n(12 non-UUP)")
ax.set_xlabel("rebalance date")
ax.set_title("F7(c): rho_bar_rest", fontsize=9)

plt.tight_layout(); plt.show()

train_mask_f7 = f7_table.index <= split_ts
test_mask_f7 = f7_table.index > split_ts

def f7_summary_row(mask_or_date):
    if isinstance(mask_or_date, pd.Timestamp):
        row = f7_table.loc[mask_or_date]
        return {
            "median_sigma_UUP": float(row["sigma_UUP"]),
            "median_sqrt_V_rest": float(row["sqrt_V_rest"]),
            "median_ratio": float(row["sqrt_V_rest"] / row["sigma_UUP"]),
            "median_rho_bar_rest": float(row["rho_bar_rest"]),
            "median_w_UUP_HRP": float(row["w_UUP_HRP"]),
            "root_singleton_share": float(row["root_singleton"]),
        }
    sub = f7_table.loc[mask_or_date]
    ratio = sub["sqrt_V_rest"] / sub["sigma_UUP"]
    return {
        "median_sigma_UUP": float(sub["sigma_UUP"].median()),
        "median_sqrt_V_rest": float(sub["sqrt_V_rest"].median()),
        "median_ratio": float(ratio.median()),
        "median_rho_bar_rest": float(sub["rho_bar_rest"].median()),
        "median_w_UUP_HRP": float(sub["w_UUP_HRP"].median()),
        "root_singleton_share": float(sub["root_singleton"].mean()),
    }

f7_summary = pd.DataFrame({
    "train (<= split)": f7_summary_row(train_mask_f7),
    "test (> split)": f7_summary_row(test_mask_f7),
    "2022-12-31": f7_summary_row(pd.Timestamp("2022-12-31")),
}).T
print(f7_summary.to_string(float_format=lambda v: f"{v:.4f}"))

## §8 TAKEAWAY

- **H1a — NOT ROBUST.** w_UUP(HRP) &lt; w_UUP(GMV) AND &lt; w_UUP(MDP) holds
  on 159/208 (76.4%) of all rebalances but only 14/40 (35.0%) of the test
  rebalances — it clears the &gt; 50%-of-208 bar but fails the
  &gt; 50%-of-test bar, so by the pre-registered count rule this is
  NOT ROBUST, not SUPPORTED. At 2022-12-31 the inequality does hold:
  w_UUP = HRP 0.5530, GMV 0.5812, MDP 0.5922, ERC 0.4836 — HRP's cash
  weight in UUP is smaller than GMV's and MDP's at that date, but the test
  window as a whole does not confirm the pattern out of sample.
- **H2b vs GMV — SUPPORTED.** &#124;w_UUP(Σ) − w_UUP(Σ_cf)&#124; for HRP is
  smaller than the same quantity for GMV on 100.0% of the full 208
  rebalances and 100.0% of the 40 test rebalances (2022-12-31 holds too).
- **H2b vs MDP — SUPPORTED.** Same comparison against MDP: 100.0% full,
  100.0% test.
- **H2b vs ERC — SUPPORTED.** Same comparison against ERC: 98.1% full,
  100.0% test. Next to these three counts: UUP is a root singleton under
  BOTH Σ and Σ_cf on 167/208 rebalances — whenever that holds, HRP's own
  &#124;Δw_UUP&#124; is exactly 0 by the root-singleton identity (see below),
  which is most of H2b's empirical content, not an estimated effect.
- **H3 vs GMV — SUPPORTED.** effN(HRP) &gt; effN(GMV) on 94.7% full, 90.0%
  test rebalances (2022-12-31 holds); median effN HRP(S)=5.03 vs
  GMV(S)=3.79.
- **H3 vs MDP — SUPPORTED.** effN(HRP) &gt; effN(MDP) on 100.0% full,
  100.0% test; median effN MDP(S)=3.35.
- **H3 vs ERC — SUPPORTED.** effN(HRP) &lt; effN(ERC) on 100.0% full,
  100.0% test; median effN ERC(S)=6.86 — HRP sits between GMV/MDP and ERC
  in concentration, as the pre-registered direction predicted.
- **H4 (HRP(S)−GMV(S)) — NULL HOLDS NARROWLY.** Δα from the direct
  diff-regression, t = 1.53.
- **H4 (HRP(S)−MDP(S)) — NULL HOLDS.** Δα t = 0.54.
- **H4 (HRP(S)−ERC(S)) — NULL HOLDS NARROWLY.** Δα t = 1.59.
- **H5 (HRP−EW, beta-driven) — NOT SUPPORTED.** The full-window paired t
  is −1.94, just short of the −2 threshold required by the pre-registered
  rule (the test-window mean is negative, −0.0357, and
  &#124;Δβ part&#124; = 0.0460 &gt; &#124;Δα&#124; = 0.0130, but the full-window t
  does not clear the bar, so H5 is not supported under the rule as
  written).
- **H6 (HRP(LW)−HRP(S), total) — NULL HOLDS.** t_full = 0.27, t_test =
  −0.31; larger-magnitude of the two is 0.31, well inside the null band —
  the Ledoit-Wolf sensitivity makes little difference to HRP's returns.
- **H6 (HRP(S)−IVP, Δα) — NULL HOLDS.** t = 1.01.
- **H6 (HRP&#91;pos&#93;(S)−HRP(S), Δα) — NULL HOLDS.** t = −0.33 — the bisection
  choice does not show up as a return difference despite the large weight
  differences documented in §3 and §5.
- **H6 (HRP&#91;ward&#93;(S)−HRP(S), Δα) — NULL HOLDS.** t = −0.51 — same
  conclusion for the linkage sensitivity selected by the pre-set recipe
  rule in §3.5.
- **H7 (forecast bias, HRP vs GMV) — SUPPORTED.** d = q_HRP − q_GMV has
  full-window t = −6.18 and a negative test-window mean (−0.0788) —
  every method's mean q is negative (full-window: ERC(S) −0.1049, EW
  −0.1885, GMV(S) −0.0356, HRP(S) −0.0999, IVP −0.1237, MDP(S) −0.0518),
  so the result is relative: GMV's ex-ante vol is the LEAST conservative
  forecast of the six, and HRP's is more conservative than GMV's by the
  printed mean d (−0.0643). The two optimized methods (GMV, MDP) have the
  least negative mean q, consistent with optimizer bias. The common
  negative shift across all methods is not explained by this and is not
  tested here.

**Descriptive (not pre-registered as directional tests):**
- **H1b (HRP vs ERC).** No direction was registered (the design-check
  snapshot was mixed). Descriptively, w_UUP(HRP) &lt; w_UUP(ERC) on 105/208
  (50.5%) full rebalances but only 3/40 (7.5%) test rebalances — essentially
  a coin flip full-sample and rare in the test window, consistent with no
  claim being registered here.
- **H2a.** UUP is a root singleton (alone on one side of the final merge)
  on 183/208 rebalances.

**Root-singleton mechanism.** At 2022-12-31 UUP's entire tree path is a
single split — (own-side n=1, other-side n=12, share to own side=0.553) —
so w_UUP(HRP) is exactly the root-singleton identity
V_rest/(σ²_UUP + V_rest), independent of UUP's correlations with the rest
of the universe; the printed check confirms
&#124;path product − w_UUP&#124; = 1.110e-16.

**Does HRP remove the corner, or just its correlation-driven part?** HRP
removes the identity's dependence on ρ for the specific asset that is a
root singleton — but the corner survives via the variance comparison
instead: at 2022-12-31, w_UUP is HRP 0.5530, GMV 0.5812, MDP 0.5922, ERC
0.4836 — all four methods still put more than 48% of the book in UUP.
HRP's cash weight is a bit smaller than GMV's and MDP's here (consistent
with H1a holding at this one date) but is not small in absolute terms, and
it is larger than ERC's.

**Concentration at that same date.** HRP's max RC share at 2022-12-31 is
0.5829 vs. ERC's 0.0769 (ERC's is 1/13 by construction, per notebook 06) —
HRP concentrates risk far more than ERC does even though both methods put
a similar fraction of NOTIONAL in UUP; HRP never targets equal risk shares
in the first place, so this is not a violated design goal, just a
difference between the two objectives.

**Root-split decomposition (F7, descriptive).** Over train rebalances (&lt;= split), median σ_UUP=0.0752, median √V_rest=0.0448, median ratio √V_rest/σ_UUP=0.5846, median ρ̄_rest=0.2409, median w_UUP(HRP)=0.2584, and UUP is a root singleton on 85.12% of train rebalances. Over test rebalances (&gt; split), median σ_UUP=0.0741, median √V_rest=0.0634, median ratio=1.0304, median ρ̄_rest=0.4440, median w_UUP(HRP)=0.5149, and the root-singleton share rises to 100.0%. At 2022-12-31: σ_UUP=0.0927, √V_rest=0.1031, ratio=1.1123, ρ̄_rest=0.4511, root singleton=1 — the maximum deviation between HRP's w_UUP and the root-singleton identity, over all root-singleton rebalances, is 2.220e-16. Between train and test the median ratio √V_rest/σ_UUP rises from 0.5846 to 1.0304 and the median ρ̄_rest from 0.2409 to 0.4440, alongside the rise in root-singleton frequency; at 2022-12-31 UUP's 55.30% weight is the single comparison of σ²_UUP against V_rest, printed above as the identity.

**CAPM alphas.** The single-strategy CAPM α/β table in §6.5 is, as in
notebooks 04–06, not pre-registered — shown for context only, with no
hypothesis attached to any individual number in it; it is Phase 2
multiple-testing material.

**No claim beyond what is tested here is made:** this notebook does not
test whether the HRP tree structure is "the right" clustering of this
universe, does not compare HRP against a risk-budgeted (non-equal)
tree-based alternative, and does not attribute UUP's persistent root
singleton to any specific macro regime.


## §9 Notes

- (a) HAP/HARP-style hierarchical allocators need an expected-return signal
  at each split, so they are return-based rather than risk-based and are
  not backtested in this notebook.
- (b) Hierarchical-clustering portfolios of the HCP type follow the tree
  down to a cluster cut and then allocate equal capital per cluster —
  HRP[tree] shares the tree-following step with this family, and the
  literature reports such portfolios as sensitive to the cluster-count
  choice.
- (c) The positional (equal-halves) bisection is the literature's original
  recursive-bisection algorithm, kept here only as a sensitivity — §1.2 and
  §5 show why tree-following is registered instead.